# Lecture 5 : SLAM and Navigaation #

In [1]:
from IPython.display import Video

## Overview

......

## Part 1) Let's Get Started!

### Lets build our workspace containing the demonstration code

#### 1. Create your workspace
``` bash
source /opt/ros/humble/setup.bash
mkdir -p ~/fra532_lecture5_ws/src
colcon build
source fra532_lecture5_ws/install/setup.bash
```

#### 2. Download and Clone Package

MIR Robot
``` bash
cd ~/fra532_lecture5_ws/

# clone mir_robot into the ros2 workspace
git clone -b humble-devel https://github.com/relffok/mir_robot src/mir_robot

# use vcs to fetch linked repos
vcs import < src/mir_robot/ros2.repos src --recursive

# use rosdep to install all dependencies (including ROS itself)
sudo apt update
sudo apt install -y python3-rosdep
rosdep update --rosdistro=humble
rosdep install --from-paths src --ignore-src -r -y --rosdistro humble

# build all packages in the workspace
cd ~/fra532_lecture5_ws
colcon build

```

You must source the workspace in each terminal you want to work in:

``` bash
source ~/fra532_lecture5_ws/install/setup.bash
```

Alternatively you can add the following line to the end of your ~/.bashrc, and then close and reopen all terminals

``` bash
echo "source ~/fra532_lecture5_ws/install/setup.bash" >> ~/.bashrc
```



``` bash
ros2 launch mir_gazebo mir_gazebo_launch.py world:=maze rviz_config_file:=$(ros2 pkg prefix mir_navigation)/share/mir_navigation/rviz/mir_nav.rviz
```

#### 3. Download worlds model file

``` bash
cd 
git clone https://github.com/aws-robotics/aws-robomaker-small-warehouse-world.git -b ros2

# build for ROS2
cd ~/fra532_lecture5_ws
rosdep install --from-paths . --ignore-src -r -y
colcon build

# run in ROS2
source install/setup.sh
ros2 launch aws_robomaker_small_warehouse_world small_warehouse.launch.py
```

#### 4. Create package for integrate MIR robot and warehouse world.

Using ROS2_pkg_cpp_py

``` bash
# Clone ROS2_pkg_cpp_py to Workspace
cd 
git clone https://github.com/tchoopojcharoen/ROS2_pkg_cpp_py.git

# Generate new package
. ROS2_pkg_cpp_py/install_pkg.bash {YOUR_WORKSPACE} {PACKAGE_NAME}
. ROS2_pkg_cpp_py/install_pkg.bash fra532_lecture5_ws fra532_nav
. ROS2_pkg_cpp_py/install_pkg.bash fra532_lecture5_ws fra532_slam
. ROS2_pkg_cpp_py/install_pkg.bash fra532_lecture5_ws fra532_gazebo

```


#### 5. Create Launch File for ....

1. ไปที่ Package fra532_gazebo

1.1 สร้าง Folder Launch

1.2 ไปที่ไฟล์ CMakeLists.txt >> Launch ใน ### INSTALL LAUNCH, ETC ###

1.3 สร้างไฟล์ sim.launch.py ใน Folder Launch

Building and Launching the Gazebo World with your ROS Applications

```python
    import os

    from ament_index_python.packages import get_package_share_directory
    from launch import LaunchDescription
    from launch.actions import IncludeLaunchDescription
    from launch.launch_description_sources import PythonLaunchDescriptionSource

    def generate_launch_description():
        warehouse_pkg_dir = get_package_share_directory('aws_robomaker_small_warehouse_world')
        warehouse_launch_path = os.path.join(warehouse_pkg_dir, 'launch')

        warehouse_world_cmd = IncludeLaunchDescription(
            PythonLaunchDescriptionSource([warehouse_launch_path, '/no_roof_small_warehouse.launch.py'])
        )

        ld = LaunchDescription()

        ld.add_action(warehouse_world_cmd)

        return ld
```

2. ใส่หุ่นยนต์

```python
    import os

    from ament_index_python.packages import get_package_share_directory
    from launch import LaunchDescription
    from launch.actions import IncludeLaunchDescription
    from launch.launch_description_sources import PythonLaunchDescriptionSource

    def generate_launch_description():
        warehouse_pkg_dir = get_package_share_directory('aws_robomaker_small_warehouse_world')
        warehouse_launch_path = os.path.join(warehouse_pkg_dir, 'launch')

        # Add Here
        mir_description_dir = get_package_share_directory('mir_description')
        mir_gazebo_dir = get_package_share_directory('mir_gazebo')
        gazebo_ros_dir = get_package_share_directory('gazebo_ros')

        
        warehouse_world_cmd = IncludeLaunchDescription(
            PythonLaunchDescriptionSource([warehouse_launch_path, '/no_roof_small_warehouse.launch.py'])
        )

        # Add Here
        spawn_robot = Node(
            package='gazebo_ros',
            executable='spawn_entity.py',
            arguments=['-entity', LaunchConfiguration('robot_name'),
                    '-topic', 'robot_description',
                    '-b'],  # bond node to gazebo model,
            namespace=LaunchConfiguration('namespace'),
            output='screen')

        ld = LaunchDescription()

        ld.add_action(warehouse_world_cmd)
        ld.add_action(spawn_robot)

        return ld
```

``` bash
cd ~/fra532_lecture5_ws
colcon build && source install/setup.bash && ros2 launch fra532_gazebo sim.launch.py 
```

<img src="media/warehouse_mir.png" alt="drawing" width="800"/>


#### 6. Create Launch File for 

1. ไปที่ Package fra532_slam
1.1 สร้าง Folder >> launch, config 
1.2 ไปที่ไฟล์ CMakeLists.txt >> launch, config ใน ### INSTALL LAUNCH, ETC ###
1.3 สร้างไฟล์ slam.launch.py ใน Folder launch
1.4 สร้างไฟล์ mapping_async.yaml ใน Folder >> config

``` bash
slam_toolbox:
  ros__parameters:

    # Plugin params
    solver_plugin: solver_plugins::CeresSolver
    ceres_linear_solver: SPARSE_NORMAL_CHOLESKY
    ceres_preconditioner: SCHUR_JACOBI
    ceres_trust_strategy: LEVENBERG_MARQUARDT
    ceres_dogleg_type: TRADITIONAL_DOGLEG
    ceres_loss_function: None

    # ROS Parameters
    odom_frame: odom
    map_frame: map
    base_frame: base_footprint
    scan_topic: /scan
    mode: mapping #localization

    # if you'd like to immediately start continuing a map at a given pose
    # or at the dock, but they are mutually exclusive, if pose is given
    # will use pose
    # map_file_name: #install/share/mir_navigation/maps/maze_incomplete
    # map_start_pose: [0.0, 0.0, 0.0]
    # map_start_at_dock: true

    debug_logging: true
    throttle_scans: 1
    transform_publish_period: 0.02 #if 0 never publishes odometry
    map_update_interval: 0.01
    resolution: 0.05
    max_laser_range: 20.0 #for rastering images
    minimum_time_interval: 0.5
    transform_timeout: 0.5
    tf_buffer_duration: 300.
    stack_size_to_use: 40000000 #// program needs a larger stack size to serialize large maps
    enable_interactive_mode: true

    # General Parameters
    use_scan_matching: true
    use_scan_barycenter: true
    minimum_travel_distance: 0.5
    minimum_travel_heading: 0.5
    scan_buffer_size: 10
    scan_buffer_maximum_scan_distance: 10.0
    link_match_minimum_response_fine: 0.1  
    link_scan_maximum_distance: 1.5
    loop_search_maximum_distance: 3.0
    do_loop_closing: true 
    loop_match_minimum_chain_size: 10           
    loop_match_maximum_variance_coarse: 3.0  
    loop_match_minimum_response_coarse: 0.35    
    loop_match_minimum_response_fine: 0.45

    # Correlation Parameters - Correlation Parameters
    correlation_search_space_dimension: 0.5
    correlation_search_space_resolution: 0.01
    correlation_search_space_smear_deviation: 0.1 

    # Correlation Parameters - Loop Closure Parameters
    loop_search_space_dimension: 8.0
    loop_search_space_resolution: 0.05
    loop_search_space_smear_deviation: 0.03

    # Scan Matcher Parameters
    distance_variance_penalty: 0.5      
    angle_variance_penalty: 1.0    

    fine_search_angle_offset: 0.00349     
    coarse_search_angle_offset: 0.349   
    coarse_angle_resolution: 0.0349        
    minimum_angle_penalty: 0.9
    minimum_distance_penalty: 0.5
    use_response_expansion: true
```

Building and Launching the SLAM
1.5 สร้าง launch file ใน Folder >> launch ชื่อว่า mapping.launch.py
```python
import os

from launch import LaunchDescription
from launch.actions import DeclareLaunchArgument, IncludeLaunchDescription, OpaqueFunction, \
    SetLaunchConfiguration
from launch.launch_description_sources import PythonLaunchDescriptionSource
from launch.conditions import IfCondition
from launch.substitutions import LaunchConfiguration
from ament_index_python.packages import get_package_share_directory


def generate_launch_description():

    mir_driver_dir = get_package_share_directory('mir_driver')
    mir_nav_dir = get_package_share_directory('mir_navigation')

    def declare_rviz_config(context):
        nav_enabled = context.launch_configurations['navigation_enabled']
        if (nav_enabled == 'true'):
            config_file = os.path.join(
                mir_nav_dir, 'rviz', 'mir_mapping_nav.rviz')
        else:
            config_file = os.path.join(mir_nav_dir, 'rviz', 'mir_mapping.rviz')
        return [SetLaunchConfiguration('rviz_config_file', config_file)]

    declare_use_sim_time_argument = DeclareLaunchArgument(
        'use_sim_time',
        default_value='false',
        description='Use simulation/Gazebo clock')

    declare_slam_params_file_cmd = DeclareLaunchArgument(
        'slam_params_file',
        default_value=os.path.join(get_package_share_directory("mir_navigation"),
                                   'config', 'mir_mapping_async.yaml'),
        description='Full path to the ROS2 parameters file to use for the slam_toolbox node')

    declare_nav_argument = DeclareLaunchArgument(
        'navigation_enabled',
        default_value='false',
        description='Use navigation for mapping')

    declare_namespace_arg = DeclareLaunchArgument(
        'namespace',
        default_value='',
        description='Namespace to push all topics into.')

    start_driver_cmd = IncludeLaunchDescription(
        PythonLaunchDescriptionSource(
            os.path.join(mir_driver_dir, 'launch', 'mir_launch.py')),
        launch_arguments={'rviz_config_file': LaunchConfiguration(
            'rviz_config_file')}.items()
    )

    launch_mapping = IncludeLaunchDescription(
        PythonLaunchDescriptionSource(
            os.path.join(mir_nav_dir, 'launch', 'include', 'mapping.py')),
        launch_arguments=[
            ('use_sim_time', LaunchConfiguration('use_sim_time')),
            ('slam_params_file', LaunchConfiguration('slam_params_file'))]
    )

    launch_navigation_if_enabled = IncludeLaunchDescription(
        PythonLaunchDescriptionSource(
            os.path.join(mir_nav_dir, 'launch', 'include', 'navigation.py')),
        condition=IfCondition(LaunchConfiguration('navigation_enabled')),
        launch_arguments={'map_subscribe_transient_local': 'true'}.items()
    )

    ld = LaunchDescription()

    ld.add_action(declare_namespace_arg)
    ld.add_action(declare_use_sim_time_argument)
    ld.add_action(declare_nav_argument)
    ld.add_action(declare_slam_params_file_cmd)
    ld.add_action(OpaqueFunction(function=declare_rviz_config))

    ld.add_action(start_driver_cmd)
    ld.add_action(launch_mapping)
    ld.add_action(launch_navigation_if_enabled)

    return ld

```

1.6 สร้าง rviz file ใน Folder >> rviz ชื่อว่า mapping.rviz

``` bash
cd ~/fra532_lecture5_ws
colcon build && source install/setup.bash && ros2 launch fra532_slam mapping.launch.py  
```

<video width="800" controls>
  <source src="media/slam.mp4" type="video/mp4">
  Your browser does not support the video tag.
</video>

#### 7. Create Launch File for Navigation

1.1 สร้าง Folder >> launch, config, rviz

1.2 ไปที่ไฟล์ CMakeLists.txt >> launch, config, rviz ใน ### INSTALL LAUNCH, ETC ###

1.3 สร้างไฟล์ navigation.launch.py ใน Folder launch

Building and Launching the Gazebo World with your ROS Applications


``` python
# Copyright (c) 2021 Samsung Research America
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import os

from ament_index_python.packages import get_package_share_directory

from launch import LaunchDescription
from launch.actions import DeclareLaunchArgument, ExecuteProcess, IncludeLaunchDescription
from launch.conditions import IfCondition
from launch.launch_description_sources import PythonLaunchDescriptionSource
from launch.substitutions import LaunchConfiguration, PythonExpression
from launch_ros.actions import Node


def generate_launch_description():
    # Get the launch directory
    nav2_bringup_dir = get_package_share_directory('nav2_bringup')
    bringup_dir = get_package_share_directory('nav2_rosdevday_2021')
    robot_model_dir = get_package_share_directory('neo_simulation2')
    warehouse_dir = get_package_share_directory('aws_robomaker_small_warehouse_world')

    nav2_launch_dir = os.path.join(nav2_bringup_dir, 'launch')
    launch_dir = os.path.join(bringup_dir, 'launch')
    rviz_config_file = os.path.join(nav2_bringup_dir, 'rviz', 'nav2_default_view.rviz')

    # Create the launch configuration variables
    slam = LaunchConfiguration('slam')
    map_yaml_file = LaunchConfiguration('map')
    use_sim_time = LaunchConfiguration('use_sim_time')
    params_file = LaunchConfiguration('params_file')

    # Launch configuration variables specific to simulation
    use_simulator = LaunchConfiguration('use_simulator')
    use_rviz = LaunchConfiguration('use_rviz')
    headless = LaunchConfiguration('headless')
    world = LaunchConfiguration('world')
    urdf = LaunchConfiguration('urdf')

    # Declare the launch arguments
    declare_slam_cmd = DeclareLaunchArgument(
        'slam',
        default_value='False',
        description='Whether run a SLAM')

    declare_map_yaml_cmd = DeclareLaunchArgument(
        'map',
        default_value=os.path.join(
            warehouse_dir, 'maps', '005', 'map.yaml'),
        description='Full path to map file to load')

    declare_use_sim_time_cmd = DeclareLaunchArgument(
        'use_sim_time',
        default_value='true',
        description='Use simulation (Gazebo) clock if true')

    declare_params_file_cmd = DeclareLaunchArgument(
        'params_file',
        default_value=os.path.join(bringup_dir, 'params', 'basic_params.yaml'),
        description='Full path to the ROS2 parameters file to use for all launched nodes')

    declare_use_simulator_cmd = DeclareLaunchArgument(
        'use_simulator',
        default_value='True',
        description='Whether to start the simulator')

    declare_use_rviz_cmd = DeclareLaunchArgument(
        'use_rviz',
        default_value='True',
        description='Whether to start RVIZ')

    declare_simulator_cmd = DeclareLaunchArgument(
        'headless',
        default_value='False',
        description='Whether to execute gzclient for the simulation frontend')

    declare_world_cmd = DeclareLaunchArgument(
        'world',
        default_value=os.path.join(
            bringup_dir, 'worlds', 'industrial_sim.world'),
        description='Full path to world model file to load')

    declare_urdf_cmd = DeclareLaunchArgument(
        'urdf',
        default_value=os.path.join(
            robot_model_dir, 'robots', 'mp_400', 'mp_400.urdf'),
        description='Full path to world model file to load')

    # Specify the actions
    start_gazebo_server_cmd = ExecuteProcess(
        condition=IfCondition(use_simulator),
        cmd=['gzserver', '-s', 'libgazebo_ros_factory.so', world],
        cwd=[warehouse_dir], output='screen')

    start_gazebo_client_cmd = ExecuteProcess(
        condition=IfCondition(PythonExpression(
            [use_simulator, ' and not ', headless])),
        cmd=['gzclient'],
        cwd=[warehouse_dir], output='screen')

    spawn_entity_cmd = Node(
        package='gazebo_ros',
        executable='spawn_entity.py',
        arguments=['-entity', 'robot',
                   '-file', urdf,
                   '-x', '3.45',
                   '-y', '2.15',
                   '-z', '0.10',
                   '-Y', '3.14'],
        output='screen')

    start_robot_state_publisher_cmd = Node(
        package='robot_state_publisher',
        executable='robot_state_publisher',
        name='robot_state_publisher',
        output='screen',
        parameters=[{'use_sim_time': use_sim_time}],
        arguments=[urdf])

    rviz_cmd = IncludeLaunchDescription(
        PythonLaunchDescriptionSource(
            os.path.join(nav2_launch_dir, 'rviz_launch.py')),
        condition=IfCondition(use_rviz),
        launch_arguments={'namespace': '',
                          'use_namespace': 'False',
                          'use_sim_time': use_sim_time,
                          'rviz_config': rviz_config_file}.items())

    bringup_cmd = IncludeLaunchDescription(
        PythonLaunchDescriptionSource(
            os.path.join(nav2_launch_dir, 'bringup_launch.py')),
        launch_arguments={'slam': slam,
                          'map': map_yaml_file,
                          'use_sim_time': use_sim_time,
                          'params_file': params_file,
                          'autostart': 'True'}.items())

    # UNCOMMENT HERE FOR KEEPOUT DEMO
    # start_lifecycle_manager_cmd = Node(
    #         package='nav2_lifecycle_manager',
    #         executable='lifecycle_manager',
    #         name='lifecycle_manager_costmap_filters',
    #         output='screen',
    #         emulate_tty=True,
    #         parameters=[{'use_sim_time': use_sim_time},
    #                     {'autostart': True},
    #                     {'node_names': ['filter_mask_server', 'costmap_filter_info_server']}])

    # start_map_server_cmd = Node(
    #         package='nav2_map_server',
    #         executable='map_server',
    #         name='filter_mask_server',
    #         output='screen',
    #         emulate_tty=True,
    #         parameters=[params_file])

    # start_costmap_filter_info_server_cmd = Node(
    #         package='nav2_map_server',
    #         executable='costmap_filter_info_server',
    #         name='costmap_filter_info_server',
    #         output='screen',
    #         emulate_tty=True,
    #         parameters=[params_file])

    # Create the launch description and populate
    ld = LaunchDescription()

    # Declare the launch options
    ld.add_action(declare_slam_cmd)
    ld.add_action(declare_map_yaml_cmd)
    ld.add_action(declare_use_sim_time_cmd)
    ld.add_action(declare_params_file_cmd)
    ld.add_action(declare_use_simulator_cmd)
    ld.add_action(declare_use_rviz_cmd)
    ld.add_action(declare_simulator_cmd)
    ld.add_action(declare_world_cmd)
    ld.add_action(declare_urdf_cmd)
    ld.add_action(start_gazebo_server_cmd)
    ld.add_action(start_gazebo_client_cmd)
    ld.add_action(spawn_entity_cmd)
    ld.add_action(start_robot_state_publisher_cmd)
    ld.add_action(rviz_cmd)
    ld.add_action(bringup_cmd)
    # UNCOMMENT HERE FOR KEEPOUT DEMO
    # ld.add_action(start_lifecycle_manager_cmd)
    # ld.add_action(start_map_server_cmd)
    # ld.add_action(start_costmap_filter_info_server_cmd)

    return ld

```

# Costmap Filters for a full application

In that last demonstrations, the robot drove under the shelving units because the robot was small enough to fit. It also traveled through outlined safety tape marking a robot-free area. Obviously in a real use-case, that would either be unsafe or a bad idea. Lets apply the new **Keepout Zones** costmap filter to this problem. We could also use the **Speed Restricted Zones** filter, but doesn't flow nicely from the last demonstration ;-).

#### Keepout Mask
The keepout mask is a file, similar to a map, that contains the mask to apply as the keepout zone. This does not have to be binary no-go zones, it can also just embed areas of higher cost that are still possible to traverse if there are no better alternatives. We've pre-generated one for this demo that makes binary no-go zones in the safety tape and under the shelves.

``` bash
sudo apt-get install gimp
```

<img src="media/map-nav.png" alt="drawing" width="200"/>